In [ ]:
##importing libraries
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from dipy.segment.mask import median_otsu
from torch.utils.data import Dataset, DataLoader
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import time

c:\Users\Acer\Desktop\Saroj_Project\IVIM_Autoencoder\ivimvnv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


In [ ]:
##pytorch details
pip show torch

Name: torch
Version: 2.8.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: c:\users\acer\desktop\saroj_project\ivim_autoencoder\ivimvnv\lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, sympy, typing-extensions
Required-by: monai, torchaudio, torchvision
Note: you may need to restart the kernel to use updated packages.


In [ ]:
##Checking torch version, cuda, GPU and memory 
print(torch.__version__)              # Should show +cu128 (not +cpu)
print(torch.cuda.is_available())      # Should be True
print(torch.cuda.get_device_name(0))  # Should show "NVIDIA GeForce GTX 1650 Ti"
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

2.8.0+cu128
True
NVIDIA GeForce GTX 1650 Ti
VRAM: 4.29 GB


In [ ]:
##Processors
import subprocess
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        check=True
    )
    print(result.stdout)
except FileNotFoundError:
    print("nvidia-smi is not installed or not on PATH.")
except subprocess.CalledProcessError as exc:
    print("nvidia-smi failed:", exc.stderr or exc)

Mon Aug  3 12:49:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.44                 Driver Version: 591.44         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 Ti   WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              3W /   30W |     235MiB /   4096MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## **DATASET**

In [7]:
##loding data
path = "C:\\Users\\Acer\\Desktop\\Saroj_Project\\IVIM_Autoencoder\\notebooks\\BRAIN"

raw_data=nib.load(path+"\\data.nii.gz")
mri_data=raw_data.get_fdata()
print(mri_data.shape)

(256, 256, 54, 21)


## **Data description**
1. H: Height of one image in the sequence
2. W: Width of one image in the sequence
3. D: Depth of images or Number of images in the sequence
4. C: Number of channels (like 3 RGB or b-values)

In [4]:
# data = torch.randn(2,3,4,4)
# data.shape
# data[0]
# plt.imshow(data[0])

In [21]:
# Load b-values from the file
with open(path+"/bvalues.bval", "r") as f:
    bvals_str = f.read().split()
bvals_arr = np.array([float(b) for b in bvals_str], dtype=np.float32)

print(f"Loaded b-values: {bvals_arr}")
print(f"Number of b-values (N_b): {len(bvals_arr)}")

Loaded b-values: [   0.   10.   20.   30.   40.   60.   80.  100.  120.  140.  160.  180.
  200.  300.  400.  500.  600.  700.  800.  900. 1000.]
Number of b-values (N_b): 21


In [ ]:
bvals_arr

## **DATA PREPROCESSING FOR SIMILAR NEIGHBOURING PIXELS EXTRACTION**

In [8]:
import numpy as np

# 1. Get raw dimensions
H, W, D, b_value = mri_data.shape 

# 2. Calculate indices for a 70/15/15 split
train_end = int(D * 0.70)
val_end = int(D * 0.85)  # 70% + 15% = 85%

# 3. Slice Raw MRI Data (Along Axis 2 / Depth)
raw_train_data = mri_data[:, :, :train_end, :]
raw_val_data   = mri_data[:, :, train_end:val_end, :]
raw_test_data  = mri_data[:, :, val_end:, :]

In [ ]:
##displaying the 2 slice of mri data over 21 b-value
fig, ax = plt.subplots(3,7, figsize=(10,6))

# Flatten the 2D array of axes into a 1D array for easier iteration
flat_axes = ax.ravel()

plot_counter = 0
# Iterate over the first two slices (depth dimension) of the MRI data
for i in range(raw_train_data[:,:,:2,:].shape[2]):
  # Iterate over all b-values
  for j in range(raw_train_data.shape[3]): # j will be 0 to 20 b-values
    if plot_counter < len(flat_axes): # Check if there are still subplots available
      current_ax = flat_axes[plot_counter]
      current_ax.imshow(raw_train_data[:,:,i,j], cmap='gray')
      current_ax.set_title(f"Slice {i}, b={j}") # Add a title for context
      current_ax.axis('off') # Turn off axis labels for cleaner display
      plot_counter += 1

plt.tight_layout() # Adjust subplot parameters for a tight layout
plt.show()

In [ ]:
img = raw_train_data[:,:,20,0]
#for standardize the pixel values
# mean = np.mean(img)
# std = np.std(img)
# img = img-mean
# img = img/std
plt.hist(img.flatten(),bins=200)
plt.show()


In [ ]:
#Slice axis 3 (b-value) at index 0, keeping all H, W, and D
b0_raw = mri_data[:, :, :, 0]  #(256, 256, 54) -> (H, W, D)

# Generate the mask using the correct 3D spatial volume
_, otsu_mask = median_otsu(b0_raw, median_radius=3, numpass=2)
brain_msk = otsu_mask.astype(np.uint8)   # (256, 256, 54) -> (H, W, D)

#Correct spatial dimension
print("Mask shape (H, W, D):", brain_msk.shape)  #(256, 256, 54)


Mask shape (H, W, D): (256, 256, 54)


In [ ]:
#Slice Brain Masks to Match
train_mask = brain_msk[:, :, :train_end]
val_mask   = brain_msk[:, :, train_end:val_end]
test_mask  = brain_msk[:, :, val_end:]

print(f'shape of train: {train_mask.shape}')
print(f'shape of val: {val_mask.shape}')
print(f'shape of test: {test_mask.shape}')

shape of train: (256, 256, 37)
shape of val: (256, 256, 8)
shape of test: (256, 256, 9)


In [ ]:
##process in the chunk to speed up
def process_chunk(padded_px, padded_b0, y, x, dy, dx,
                    center_flat_idx, target_slots, b_value):
    """
    Broadcasting Technique:
      Vectorized core, run on one chunk of pixels at a time so peak memory
    stays bounded regardless of total voxel count N.
    """

    n = len(y)

    ##Broadcasting technique to speed up the calculation
    #Build 5x5 neighborhood (25 positions per pixel) fetch using 5x5 window
    yy = y[:, None] + dy[None, :]
    xx = x[:, None] + dx[None, :]

    ##similar neighbouring pixels calculation
    b0_neighbors = padded_b0[yy, xx]              # (n, 25)
    center_b0 = padded_b0[y, x]                     # (n,)
    similarity = np.abs(center_b0[:, None] - b0_neighbors)  # (n, 25)

    ##a masking array of 25 boolean coordinate
    ##removing center coordinate from the extracted 5x5 similarity matrix
    keep = np.ones(25, dtype=bool)
    keep[center_flat_idx] = False

    ##indexing drops/remove center from the column where keep==false
    # Drops the center coordinates
    similarity = similarity[:, keep] # (n, 24)
    yy, xx = yy[:, keep], xx[:, keep]

    #sort the python list.sort() 
    #order indices says which column to pick per row  
    order = np.argsort(similarity, axis=1, kind='stable')  # (n, 24)
    top8_indices = order[:, :8]

    ##select values from an array along a specified axis using index arrays
    yy_top = np.take_along_axis(yy, top8_indices, axis=1)      # (n, 8)
    xx_top = np.take_along_axis(xx, top8_indices, axis=1)
    full_top8 = padded_px[:, yy_top, xx_top]  # (b_value, n, 8)

    ##initialize the 3x3 patch of most similar pixels
    block = np.zeros((n, 3, 3, b_value), dtype=np.float32)
    center_vals = padded_px[:, y, x]             # (b_value, n)
    block[:, 1, 1, :] = center_vals.T

    vals = np.transpose(full_top8, (1, 2, 0))           # (n, 8, b_value)
    block[:, target_slots[:, 0], target_slots[:, 1], :] = vals

    return block

In [ ]:
##extracting the similar neighbouring pixels patches 
def similar_neighborhood_patch(data, brain_masker, chunk_size=20000):
    H, W, D, b_value = data.shape

    y_factor = 2.0 / (H - 1) if H > 1 else 1.0
    x_factor = 2.0 / (W - 1) if W > 1 else 1.0

    similar_patch, center_coord, norm_cent_coord  = [], [], []

    #Looping over each slices
    for i in range(D):
      slices_data = data[:, :, i, :]

      mri_slice = np.transpose(slices_data, (2, 0, 1))  # (b, H, W)
      
      #brain mask slicing (no transpose needed for 2D)
      brain_mask_slice = brain_masker[:, :, i]  # (H, W)

      #Threshold
      #Finds the actual intensity mean of the b=0 brain pixels, not mask density
      b0_image = mri_slice[0]
      b0_img_mean = np.mean(b0_image[brain_mask_slice > 0]) if np.any(brain_mask_slice > 0) else 0
      b0_threshold = 0.50 * b0_img_mean

      # Find indices based on corrected 2D mask and threshold
      brain_indices = np.argwhere((brain_mask_slice > 0) & (b0_image >= b0_threshold))
      N = len(brain_indices)
      print(f"Slice {i:02d}/{D-1}: Processing {N} brain pixels to build 2D similarity blocks")

      ##if number of voxels is zero continue calculation
      if N == 0:
          continue
      
      ##padding tensors
      ##padded signal at b-value=0
      padded_px = np.pad(mri_slice, ((0, 0), (2, 2), (2, 2)), mode='constant', constant_values=0)
      padded_b0 = padded_px[0]


      ##x and y coordinates of brain indices with padding
      y_all = (brain_indices[:, 0] + 2).astype(np.int32)
      x_all = (brain_indices[:, 1] + 2).astype(np.int32)

      ##array ranging from -2 to 3, for generating a 5x5 window: array([-2, -1,  0,  1,  2])
      ##5x5 window matrix
      ##flatten them in 1d array
      ##offset coordinate index for central pixel
      offs = np.arange(-2, 3, dtype=np.int32)
      dy, dx = np.meshgrid(offs, offs, indexing='ij')
      dy, dx = dy.ravel(), dx.ravel()
      center_flat_idx = 12  #(0,0) offset's position in the flatten 25-length grid

      ##initialize the empty neighbors slot to store top neighbor pixels later in calculation
      top_neighbors_slot = np.array([
          (a, b) for a in range(3) for b in range(3) if not (a == 1 and b == 1)
          ])

      ##initialize the empty similar neighbourhood block array
      similar_neig_block = np.zeros((N, 3, 3, b_value), dtype=np.float32)

      ##actual similar neighbourhood data calculated in chunks and store in pre-defined similar_neig_block
      for start in range(0, N, chunk_size):
          end = min(start + chunk_size, N)
          similar_neig_block[start:end] = process_chunk(padded_px, padded_b0, y_all[start:end], x_all[start:end],
                                                    dy, dx, center_flat_idx, top_neighbors_slot, b_value)

      center_coords = list(map(tuple, brain_indices))
      y0 = brain_indices[:, 0] * y_factor - 1.0
      x0 = brain_indices[:, 1] * x_factor - 1.0
      center_norm_coord = list(zip(y0.tolist(), x0.tolist()))

      similar_patch.append(similar_neig_block)
      center_coord.append(center_coords)
      norm_cent_coord.append(center_norm_coord)
      
    print("Processing completed.")
    return similar_patch, center_coord, norm_cent_coord


In [ ]:
#Run Patches extraction function independently on each chunk
print("Extracting Training Patches")
train_patches, _, train_norm_coords = similar_neighborhood_patch(raw_train_data, train_mask)

print("\nExtracting Validation Patches")
val_patches, _, val_norm_coords = similar_neighborhood_patch(raw_val_data, val_mask)

print("\nExtracting Testing Patches")
test_patches, test_coords, test_norm_coords = similar_neighborhood_patch(raw_test_data, test_mask)

#Concatenate slices into monolithic numpy arrays
X_train = np.concatenate(train_patches, axis=0)  # (N_train, 3, 3, b_value)
X_val   = np.concatenate(val_patches, axis=0)    # (N_val, 3, 3, b_value)
X_test  = np.concatenate(test_patches, axis=0)   # (N_test, 3, 3, b_value)

# Flatten the coordinate lists for conditional context
coord_train = np.array([c for slice_c in train_norm_coords for c in slice_c], dtype=np.float32)
coord_val   = np.array([c for slice_c in val_norm_coords for c in slice_c], dtype=np.float32)
coord_test  = np.array([c for slice_c in test_norm_coords for c in slice_c], dtype=np.float32)


--- Extracting Training Patches ---
Slice 00/36 | Processing 1100 brain pixels to build 2D similarity blocks...
Slice 01/36 | Processing 1311 brain pixels to build 2D similarity blocks...
Slice 02/36 | Processing 1669 brain pixels to build 2D similarity blocks...
Slice 03/36 | Processing 2411 brain pixels to build 2D similarity blocks...
Slice 04/36 | Processing 3326 brain pixels to build 2D similarity blocks...
Slice 05/36 | Processing 4100 brain pixels to build 2D similarity blocks...
Slice 06/36 | Processing 4728 brain pixels to build 2D similarity blocks...
Slice 07/36 | Processing 5202 brain pixels to build 2D similarity blocks...
Slice 08/36 | Processing 5564 brain pixels to build 2D similarity blocks...
Slice 09/36 | Processing 5959 brain pixels to build 2D similarity blocks...
Slice 10/36 | Processing 6853 brain pixels to build 2D similarity blocks...
Slice 11/36 | Processing 8110 brain pixels to build 2D similarity blocks...
Slice 12/36 | Processing 9335 brain pixels to build 

In [ ]:
##display 3x3 block
plt.imshow(X_train[0][0,:,:,0], cmap='gray') # Display the signal at 21 b-value of the first block's middle depth slice
plt.show()

X_train[0] ##shape=(119110, 3, 3, 21)

x=torch.from_numpy(X_train[0]).float()
x.shape


##shape=(119110, 3, 3, 21) -----> shape=(119110, 21, 3, 3) ----> Idx shape=(0, 3, 1, 2)
x1=x.permute(2, 0, 1) 
x1.shape


print(x1[:,0:1, :, : ])
x1[:, 0:1, :, : ].shape ##torch.Size([21, 3, 3])


x1[:,0:1,:,:][0].shape

##b0_image
plt.imshow(x1[:, 0:1, :, :][0], cmap='gray')
plt.show()


# Visualise the highest b-value (e.g., index -1), with non-zero b-value
plt.imshow(x1[0, -1, :, :], cmap="gray", vmin=0, vmax=1)
plt.colorbar()
plt.show()

In [ ]:
##A custom data creator function
class DiffusionDataset(Dataset):
    def __init__(self, similar_patches, condition):
        """
        similar_patches: NumPy array of shape (N, 3, 3, b_values)
        condition (list of coordinates): NumPy array of shape (N, 2) 
        """
        # data is a list of slices, concatenate it into a monolithic array 
        # inside (__init__) to protect from list indexing bugs
        ##checking whether it is a list or array
        if isinstance(similar_patches, list):
            self.patches = np.concatenate(similar_patches, axis=0)
        else:
            self.patches = similar_patches

        # Convert the entire condition dataset to a PyTorch tensor at initialization. 
        # This avoids slow memory re-allocations in __getitem__
        self.condition = torch.tensor(np.array(condition), dtype=torch.float32)

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        #convert the individual patch array to a tensor
        x = torch.from_numpy(self.patches[idx]).float()

        # checking dimension for reshaping
        if x.dim() == 3:
            x = x.permute(2, 0, 1)      #(3, 3, b_values) -> (b_values, 3, 3)
        elif x.dim() == 4:
            x = x.permute(0, 3, 1, 2)   #(N, 3, 3, b_values) -> (N, b_values, 3, 3)
        else:
            raise ValueError(f"Unexpected tensor shape {tuple(x.shape)}")

        # Clean, fast index fetch for the condition vector
        y = self.condition[idx]
        return x, y



In [ ]:
#INSTANTIATE THE TORCH DATALOADERS
# Voxel-wise/Patch regression networks process data fast. A large batch size 
# improves training speed and stabilizes the CVAE latent space optimization.
BATCH_SIZE = 1024  

train_dataset = DiffusionDataset(X_train, coord_train)
val_dataset   = DiffusionDataset(X_val, coord_val)
test_dataset  = DiffusionDataset(X_test, coord_test)

# Training, Validation and Testing data loaders
#Shuffle=True is strictly applied only to training data
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)


print(f"Data Loaders successfully created:")
print(f"Train Batches:      {len(train_loader)} (Total patches: {len(train_dataset)})")
print(f"Validation Batches: {len(val_loader)} (Total patches: {len(val_dataset)})")
print(f"Testing Batches:    {len(test_loader)} (Total patches: {len(test_dataset)})")



Data Loaders successfully created:
  -> Train Batches:      473 (Total patches: 485126)
  -> Validation Batches: 135 (Total patches: 137341)
  -> Testing Batches:    83 (Total patches: 84705)


In [ ]:
# Fetch one test batch and check
for test_x, test_coord in test_loader:
    print(f"Batch X Shape (Expects [4, b_values, 3, 3]): {test_x.shape}")
    print(f"Batch Y Shape (Expects [4, 2]):           {test_coord.shape}")
    break


Batch X Shape (Expects [4, b_values, 3, 3]): torch.Size([1024, 21, 3, 3])
Batch Y Shape (Expects [4, 2]):           torch.Size([1024, 2])


## **PHYSIOLOGICAL BI-EXPONENTIAL SIGNAL EQUATION**

In [17]:
##Physics IVIM signal function
def ivim_biex_signal(theta, b_vals):
    ##IVIM parameters
    D = theta[:, 0:1]
    Ds = theta[:, 1:2]
    f = theta[:, 2:3]

    #b values broadcastable if b-value is 1D array
    if b_vals.dim()<=2:
        b = b_vals.view(1,-1, 1, 1)
    else:
        b = b_vals.reshape(1, -1, 1, 1)
    
    ##perfusion signal due to the flow of blood
    s_perf = f*torch.exp(-b * Ds)
    ##True diffusion signal due to water molecules in tissues
    s_diff = (1-f)*torch.exp(-b*D)
    ##bi-exponential signal
    s_sum = s_perf + s_diff
    return s_sum, s_perf, s_diff

## **MODEL ARCHITECTURE**

In [ ]:
##Epoch
EPOCHS = 100
INPUT_CHANNEL = len(bvals_arr)
LATENT_DIM = 64
COORD_DIM = 2
LEARNING_RATE = 1e-8
KL_TAR = 0.01
PARAM_WEIGHT = 0.1
WARMUP_EPS = 10
ANNEAL_EPS = 30

In [ ]:
##Physics informed IVIM cVAE
class PhysicsCVAE(nn.Module):
    """
    Physics-Informed Conditional Variational Autoencoder (PIcVAE) for IVIM parameter estimation.

    Encoder: 2D Convolutional Layer for 3x3 patches
    Decoder: z+Coordinates per pixel
    """
    def __init__(self, in_channel:int, bvalues: torch.Tensor, latent_dim:int, coord_dim:int, bvals=None):
        """
        PARAMETERS:
        in_channel: Input channel based on the number of b-values.
        latent_dim: Dimensionality of z.
        """
        super().__init__()
        self.latent_dim = latent_dim
        self.coord_dim = coord_dim

        ##registering b-values as buffer parameters
        self.register_buffer("bvalues", bvalues.view(1, -1, 1, 1)) 

        ##Encoder layer
        self.encoder_net = nn.Sequential(
            ##first conv layer
            nn.Conv2d(in_channel, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(0.25),

            ##second conv layer
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(0.10),

            ##third conv layer
            nn.Conv2d(128, 96, kernel_size=3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU()
        )

        # Flattened size: 32 filters * depth * H=1 * W=1
        self.encoder_fc = nn.Sequential(
            nn.Linear(96*3*3, 64),
            nn.ReLU(),
        )

        ##mean and variance for latent space
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)

        ##
        self.decoder_net = nn.Sequential(
            nn.Linear(latent_dim + self.coord_dim, 128), 
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 3*3*3) ##3 parameters
        )
        
    # Physical parameter constraints
    @staticmethod
    def physics_constraint(raw_params):
        """
        Split (B, 27) → 3 param maps with physical constraints.
        Returns: theta (B, 3, 3, 3) stacked as [D, D*, f]
                Individual: D (B,1,3,3), Ds (B,1,3,3), f (B,1,3,3)
        """
        ## Physiological Constant ranges
        D_MIN,  D_MAX   = 0.0001, 0.004 #Water restriction boundaries
        Ds_MIN, Ds_MAX  = 0.005, 0.020 #High-velocity capillary perfusion micro-circulation
        f_MIN,  f_MAX   = 0.000, 0.300 #Capillary plasma fraction cap (50%)

        ##splitting the raw params into 3 groups
        d_chunk, ds_chunk, f_chunk = torch.chunk(raw_params, chunks=3, dim=1)

        #Reshape to 2D spatial patches: Each becomes (B, 1, 3, 3)
        d_raw = d_chunk.view(-1, 1, 3, 3)
        ds_raw = ds_chunk.view(-1, 1, 3, 3)
        f_raw = f_chunk.view(-1, 1, 3, 3)

        ##IVIM THREE PARAMETERS
        ##True diffusion
        ##True and Pseudo - diffusion value is capped at maximum of 1.0
        t_D = torch.clamp(F.softplus(d_raw), max=1.0)

        ##Perfusion - pseudo-diffusion
        p_D = torch.clamp(F.softplus(ds_raw), max=1.0)

        ##perfusion fraction: Perfusion fraction remains a standard native sigmoid percentage 
        p_f = torch.sigmoid(f_raw)

        ##Scaling the three parameters using biophysiological constants 
        D  = D_MIN + t_D*(D_MAX - D_MIN)
        Ds = Ds_MIN + p_D*(Ds_MAX - Ds_MIN)
        f  = f_MIN + p_f*(f_MAX - f_MIN)

        ##Stacking parameters safely back into the (B, 3, 3, 3) shape
        theta_params = torch.cat([D, Ds, f], dim=1)

        return D, Ds, f, theta_params

    ## CVAE methods
    ##ENCODER
    def Encode(self, x):
        "The x has (batch, b-value, H, W) that generates the mu, logvar"
        h1 = self.encoder_net(x)
        h2 = h1.view(h1.size(0), -1) ##keep the batch dimension as it is, and flatten everything else.
        h3 = self.encoder_fc(h2)
        mu = self.fc_mu(h3)
        logvar = self.fc_logvar(h3)
        return mu, logvar

    ##LATENT SPACE
    ##Appying reparameterization trick: way to sample the latent variable(z) from the distribution of N(mu, variance)
    def reparameterize(self, mu, logvar):
        ##standard deviation from log-variance
        std = torch.exp(0.5 * logvar) 
        ##epsilon is a noise variable with randomness on it
        eps = torch.randn_like(std) 
        ##reparameterized sample
        z = mu + eps*std
        return z
    
    ##DECODER
    def Decode(self, z_sample, coord):
        """
        z_sample: (B, latent_dim)
        coord: (B,2)
        Returns: signal(B, b-values, 3,3)
        """
        ##decoder input
        de_input = torch.cat([z_sample, coord], dim=1) ##(B, latent_dim+2)
        raw_pars = self.decoder_net(de_input) ##(B, 27)

        ##apply physical constraint
        D, Ds, f, theta_par = self.physics_constraint(raw_pars)

        ##feeding into the IVIM bi-exponential function
        s_total, s_perf, s_diff = ivim_biex_signal(theta=theta_par, b_vals=self.bvalues)

        return s_total, (s_total, s_perf, s_diff), (theta_par, D, Ds, f)

    ##FORWARD PASS
    def forward(self, x, c):
        """
        x: (B, b-value, 3, 3)
        coord: (B, 2)
        returns: recon, mu, logvar, component, IVIM_parameters
        """
        mu, log_var = self.Encode(x)
        z = self.reparameterize(mu, log_var)
        recon, components, parameters = self.Decode(z, c)
        return recon, mu, log_var, components, parameters

    #RECONSTRUCTION LOSS WITH KL-DIVERGENCE
    def cvae_loss(self, recons, x, mu, logvar, params=None, kl_weight=1.0, param_reg_weight=0.01):
        # Critical check for broken data feeds
        if torch.isnan(x).any() or torch.isnan(recons).any():
            print("NaN detected in input signals or reconstruction layer before loss calculation")

        ##Batch size
        batch_size = x.size(0)

        #Reconstruction Loss
        recon_loss = F.mse_loss(recons, x, reduction='mean')

        #KL Divergence (Stabilized logvar constraints)
        # Clamp logvar to prevent extreme values from causing inf/nan during logvar.exp()
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        kl_loss = kl_loss / batch_size

        ##Total loss
        loss = recon_loss + (kl_weight * kl_loss)

        #Physics Constraints
        if param_reg_weight > 0 and params is not None:
            theta, D, Ds, f = params

            f_prior = 0.20  
            f_reg = ((f - f_prior) ** 2).sum() / batch_size

            # Added safety margin to avoid compounding errors
            diff_reg = (F.relu(D - Ds * 0.30 + 1e-7) ** 2).sum() / batch_size

            smooth_d_v  = ((D[:, :, 1:, :] - D[:, :, :-1, :]) ** 2).sum()
            smooth_ds_v = ((Ds[:, :, 1:, :] - Ds[:, :, :-1, :]) ** 2).sum()
            smooth_d_h  = ((D[:, :, :, 1:] - D[:, :, :, :-1]) ** 2).sum()
            smooth_ds_h = ((Ds[:, :, :, 1:] - Ds[:, :, :, :-1]) ** 2).sum()
            
            smoothness = (smooth_d_v + smooth_ds_v + smooth_d_h + smooth_ds_h) / batch_size

            total_physics_penalty = (f_reg * 1.0) + (diff_reg * 10.0) + (smoothness * 0.1)
            loss += param_reg_weight * total_physics_penalty

        # Emergency check: If anything escapes as NaN, force a small valid scalar to keep training alive
        if torch.isnan(loss):
            loss = torch.tensor(1.0, requires_grad=True).to(x.device)

        return loss, recon_loss, kl_loss


## **MODEL LOSS FUNCTION**

In [ ]:
##computing kl weights to optimize
def compute_kl_weight(epoch, warmup_epochs, anneal_epochs, kl_target):
    if epoch < warmup_epochs:
        return 0.0
    elif epoch < (warmup_epochs + anneal_epochs):
        progress = (epoch - warmup_epochs) / anneal_epochs
        return progress * kl_target
    else:
        return kl_target

##training function
def execute_cvae_training(model, train_loader, val_loader, num_epochs=100, lr=1e-4  , device="cuda"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    ##constants 
    KL_TARGET = KL_TAR         
    PARAM_REG_WEIGHT = PARAM_WEIGHT ## 0.05    
    WARMUP_EPOCHS = WARMUP_EPS ##15       
    ANNEAL_EPOCHS = ANNEAL_EPS  ##35      

    print(f"Model Training Started...")
    print("_" * 100)

    for epoch in range(num_epochs):
        #TRAINING LOOP 
        model.train()
        ## KL annealing for KL weights
        current_kl_weight = compute_kl_weight(epoch, WARMUP_EPOCHS, ANNEAL_EPOCHS, KL_TARGET)

        ##training metrics
        train_loss_accum = 0.0 
        train_mse_accum = 0.0
        train_kl_accum = 0.0

        ##iterate over batches
        ## x_batch is image pixels, coord_batch is coordinates of spatial data
        for x_batch, coord_batch in train_loader:
            x_batch, coord_batch = x_batch.to(device), coord_batch.to(device)

            ##reset all gradients to zeros that is stored in the model's parameters
            optimizer.zero_grad()

            ##performing forward pass 
            recon_signals, mu, logvar, _, params = model(x_batch, coord_batch)

            ##computing loss
            loss, recon_loss, kl_loss = model.cvae_loss(
                recons=recon_signals, x=x_batch, mu=mu, logvar=logvar,
                params=params, kl_weight=current_kl_weight, param_reg_weight=PARAM_REG_WEIGHT
            )

            ##weight updates: backpropagation 
            loss.backward()
            ##computes total L2 norm and clips all the gradients to prevent them from the gradient exploding problem
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            ##Updates the model's parameters using the clipped gradients
            optimizer.step()
            
            train_loss_accum += loss.item()
            train_mse_accum += recon_loss.item()
            train_kl_accum += kl_loss.item()

        avg_train_loss = train_loss_accum / len(train_loader)
        avg_train_mse  = train_mse_accum / len(train_loader)
        avg_train_kl   = train_kl_accum / len(train_loader)

        #VALIDATION LOOP
        model.eval()
        val_loss_accum = 0.0
        val_mse_accum = 0.0
        val_kl_accum = 0.0
        val_mae_accum = 0.0

        ##no weight updates required
        with torch.no_grad():
            ##looping over validation batches
            for x_val, coord_val in val_loader:
                x_val, coord_val = x_val.to(device), coord_val.to(device)

                ##forward pass on validation data
                recon_v, mu_v, logvar_v, _, params_v = model(x_val, coord_val)

                ##computing loss
                v_loss, v_recon, v_kl = model.cvae_loss(
                    recons=recon_v, x=x_val, mu=mu_v, logvar=logvar_v, params=params_v,
                    kl_weight=current_kl_weight, param_reg_weight=PARAM_REG_WEIGHT
                )

                ##Mean Absolute Error (MAE) between reconstruction and input validation
                mae_metric = torch.mean(torch.abs(recon_v - x_val))
                
                val_loss_accum  += v_loss.item()
                val_mse_accum   += v_recon.item()
                val_kl_accum    += v_kl.item()
                val_mae_accum   += mae_metric.item()
                    
        avg_val_loss = val_loss_accum / len(val_loader)
        avg_val_mse  = val_mse_accum / len(val_loader)
        avg_val_kl   = val_kl_accum / len(val_loader)
        avg_val_mae  = val_mae_accum / len(val_loader)
        
        avg_val_rmse = np.sqrt(avg_val_mse)

        #prints metrices
        print(f"Epoch [{epoch+1:03d}/{num_epochs:03d}] | KL-Wt: {current_kl_weight:.1e}| "
              f"Train loss: {avg_train_loss:.4f}, MSE: {avg_train_mse:.4f}|| "
              f"Val loss: {avg_val_loss:.4f}, MSE: {avg_val_mse:.4f}, RMSE: {avg_val_rmse:.4f}, MAE: {avg_val_mae:.4f}")

    print("_" * 110)
    return model


## **THREE IVIM PARAMETER EXTRACTOR**

In [19]:
##IVIM parameters; D, D*, f
def generate_IVIM_params(model, mri_brain_data, device = "cuda"):
    model.eval()
    ##initialize
    all_D_maps = []
    all_Ds_maps = []
    all_f_maps = []

    ##no weights updates
    with torch.no_grad():
        for x_batch, x_coords in mri_brain_data:
            x_batch, x_coords = x_batch.to(device), x_coords.to(device)

            ##forward pass to extract spatial aware latent variable
            _, _, _, _,parameters = model(x_batch, x_coords)

            ##unpacking the parameters from tuple
            theta, D, Ds, f = parameters


            ##squeeze into 2D from 4D shape, detach and move to cpu
            D_params = D.squeeze(1).cpu().numpy()
            Ds_params = Ds.squeeze(1).cpu().numpy()
            f_params = f.squeeze(1).cpu().numpy()

            ##store in list 
            all_D_maps.append(D_params)
            all_Ds_maps.append(Ds_params)
            all_f_maps.append(f_params)

    ##concatenating all parameters
    D   = np.concatenate(all_D_maps, axis=0)
    Ds  = np.concatenate(all_Ds_maps, axis=0)
    f   = np.concatenate(all_f_maps, axis=0)

    return D, Ds, f

## **MODEL TRAINING PIPELINE**

In [ ]:
#Trigger localized sanity-check execution run
#Set to epoch on CPU to verify syntax connectivity
bvalues_tensor = torch.from_numpy(bvals_arr.astype(np.float32)).float()

##Model
cvae_model = PhysicsCVAE(
    in_channel=INPUT_CHANNEL,  # 21 b-values
    bvalues=bvalues_tensor,
    latent_dim=LATENT_DIM,
    coord_dim=COORD_DIM  # 2D normalized coordinates
)

# Move model to device first
cvae_model.to(device)

# Run the sanity-check training pass
execute_cvae_training(
    model=cvae_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    device=device.type,
)

Model Training Started...
____________________________________________________________________________________________________
Epoch [001/100] | KL-Wt: 0.0e+00| Train loss: 19126319.8858, MSE: 19126319.8858|| Val loss: 23485681.6889, MSE: 23485681.6889, RMSE: 4846.2028, MAE: 4504.9019
Epoch [002/100] | KL-Wt: 0.0e+00| Train loss: 19125773.8478, MSE: 19125773.8478|| Val loss: 23485680.8000, MSE: 23485680.8000, RMSE: 4846.2027, MAE: 4504.9018
Epoch [003/100] | KL-Wt: 0.0e+00| Train loss: 19126435.1966, MSE: 19126435.1966|| Val loss: 23485680.1778, MSE: 23485680.1778, RMSE: 4846.2027, MAE: 4504.9017
Epoch [004/100] | KL-Wt: 0.0e+00| Train loss: 19126419.3869, MSE: 19126419.3869|| Val loss: 23485679.8519, MSE: 23485679.8519, RMSE: 4846.2026, MAE: 4504.9016
Epoch [005/100] | KL-Wt: 0.0e+00| Train loss: 19125031.9281, MSE: 19125031.9281|| Val loss: 23485678.7111, MSE: 23485678.7111, RMSE: 4846.2025, MAE: 4504.9016
Epoch [006/100] | KL-Wt: 0.0e+00| Train loss: 19126459.5645, MSE: 19126459.564

PhysicsCVAE(
  (encoder_net): Sequential(
    (0): Conv2d(21, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout2d(p=0.25, inplace=False)
    (4): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout2d(p=0.1, inplace=False)
    (8): Conv2d(128, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
  )
  (encoder_fc): Sequential(
    (0): Linear(in_features=864, out_features=64, bias=True)
    (1): ReLU()
  )
  (fc_mu): Linear(in_features=64, out_features=64, bias=True)
  (fc_logvar): Linear(in_features=64, out_features=64, bias=True)
  (decoder_net): Sequential(
    (0): Linear(in_features=66, out_features=128, bias=Tru

In [30]:
# Extract IVIM parameters from the trained model
D, Ds, f = generate_IVIM_params(cvae_model, train_loader, device=device.type)

print(f"D shape: {D.shape}, Ds shape: {Ds.shape}, f shape: {f.shape}")


D shape: (484352, 3, 3), Ds shape: (484352, 3, 3), f shape: (484352, 3, 3)


## **RECONSTRUCTION OF IMAGE WITH PARAMETERS**

In [40]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# --- 1. Convert tensors to numpy (if applicable) ---
if hasattr(D_vals, 'cpu'): D_vals = D_vals.cpu().numpy()
if hasattr(Ds_vals, 'cpu'): Ds_vals = Ds_vals.cpu().numpy()
if hasattr(f_vals, 'cpu'): f_vals = f_vals.cpu().numpy()

# --- Reconstruct 3D maps (reducing 3x3 to scalar) ---
D_map = np.zeros(brain_msk.shape)
Ds_map = np.zeros(brain_msk.shape)
f_map = np.zeros(brain_msk.shape)

pointer = 0
# Safe batch size retrieval
num_samples = x_batch.size(0) if hasattr(x_batch, 'size') else len(x_batch)

for i in range(num_samples):
    if pointer >= len(center_coords):
        break
    z, y, x = center_coords[pointer]
    
    # FIX: Reduce the (3, 3) matrix to a scalar.
    # Option A: Use np.mean() to average the 3x3 matrix.
    # Option B: Use D_vals[i][1, 1] if you specifically want the center pixel of a patch.
    d_val = np.mean(D_vals[i]) if D_vals.ndim == 3 else D_vals[i]
    ds_val = np.mean(Ds_vals[i]) if Ds_vals.ndim == 3 else Ds_vals[i]
    f_val = np.mean(f_vals[i]) if f_vals.ndim == 3 else f_vals[i]

    D_map[z, y, x] = d_val
    Ds_map[z, y, x] = ds_val
    f_map[z, y, x] = f_val
    pointer += 1

# --- Load B0 and mask ---
anat = np.array(b0_raw.get_fdata() if hasattr(b0_raw, 'get_fdata') else b0_raw)
if anat.ndim == 4:
    anat = anat[..., 0]   # take first volume if 4D

# Choose a central slice
slice_idx = anat.shape[2] // 2
brain_bg = anat[..., slice_idx]
brain_mask_2d = (brain_msk[..., slice_idx] > 0) if brain_msk.ndim == 3 else (brain_msk > 0)

# Extract 2D slices and mask them
def mask_and_clip(data_3d, mask_2d, percentile=95):
    slice_2d = data_3d[..., slice_idx]
    masked = np.where(mask_2d, slice_2d, np.nan)
    
    # FIX: Safe percentile calculation to avoid RuntimeWarning if slice is empty (all NaNs)
    valid_vals = masked[~np.isnan(masked)]
    if len(valid_vals) == 0:
        return masked
        
    vmin = 0
    vmax = np.percentile(valid_vals, percentile)
    return np.clip(masked, vmin, vmax)

D_slice = mask_and_clip(D_map, brain_mask_2d)
Ds_slice = mask_and_clip(Ds_map, brain_mask_2d)
f_slice = mask_and_clip(f_map, brain_mask_2d)

# --- Plotting ---
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

# Panel 1: original image
axes[0].imshow(brain_bg, cmap='gray')
axes[0].set_title('Original B0')
axes[0].axis('off')

# Panels 2-4: overlays
overlays = [
    ('D (mm²/s)', D_slice, 'YlOrRd'),
    ('Ds (mm²/s)', Ds_slice, 'viridis'),
    ('f (fraction)', f_slice, 'copper')
]

for ax, (title, data, cmap) in zip(axes[1:], overlays):
    # Show background
    ax.imshow(brain_bg, cmap='gray')
    
    # FIX: Safe vmax calculation to prevent crashes on empty slices
    valid_data = data[~np.isnan(data)]
    vmax = np.percentile(valid_data, 99) if len(valid_data) > 0 else 1.0
    
    # Show overlay
    im = ax.imshow(data, cmap=cmap, alpha=0.7, vmin=0, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
    
    # Add a colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

NameError: name 'center_coords' is not defined